# Mudcard
- **When the gamma is very large, why the Gaussian SVM predicts the probability of the points in most of the area behind the points (area near each of the make moons points) to be nearly 0.5?**
    - When gamma is large, each point in the training set is replaced by a very narrow gaussian so the the predicted probabilities are far from the baseline (fraction of class 1 points) only near the points
    - Everywhere else, the predicted probability is close to the baseline (fraction of class 1 points)
- **Why prediction varies smoothly with the feature values matter?**
    - It can be a secondary consideration
    - It can be introduced because of physical or domain constraints for example
        - If you try to predict speed, temperature, or altitude, etc, those quantities are inherently smooth and differentiable
        - you don't want predictions that do not respect physical continuity

# A note on imbalanced datasets
- we learnt that a classification problem is imbalanced if more than 90-95% of the points belong to one class (class 0) and only a small fraction of the points belong to the other class (class 1)
    - fraud detection
    - sick or not sick (usually by far most people are not sick)
- we learnt to not use a metric that relies on the True Negatives in the confusion matrix
    - no accuracy or ROC
    - use f_beta or the precision-recall curve instead

## What else can I do if I have an imbalanced dataset?
- most (but not all) classification algorithms we covered have a parameter called `class_weight` which allows you to assign more weight to the class 1 point
    - a misclassified class 1 point will contribute more to the cost function than a misclassified class 0 point
    - read the manual on `class_weight` because the different algorithms have slightly different definitions for this parameter
    - usually you can use `None`, `balanced`, or manually define what the class weight should be
    - it is worthwhile to tune this parameter if you have an imbalanced dataset
- resample/augment the dataset
    - SMOTE (Synthetic Minority Over-sampling Technique), see the [paper](https://arxiv.org/abs/1106.1813)
    - to improve the balance of the problem, new class 1 examples are synthesized from the existing examples
    - be careful though!
        - while resampling improves the balance of the dataset, the results of the model can be misleading
        - when you deploy the model, the incoming data will be as imbalanced as the original data

## Misleading results with resampling
- let's assume you have an imbalanced dataset with 99% of points in class 0 and 1% of points in class 1
- you resample it such that the improved class balance is 50-50
- here the confusion matrix of the trained model:

<table>
    <tr>
        <td colspan="2" rowspan="2"></td>
        <td colspan="2">Predicted class</td>			
    </tr>
    <tr>
        <td>Predicted Negative (0)</td>
        <td>Predicted Positive (1)</td>
    </tr>
    <tr>
        <td rowspan="2">Actual class</td>
        <td>Condition Negative (0)</td>
        <td><b>True Negative (TN): 45%</b></td>
        <td><b>False Positive (FP): 5%</b></td>
    </tr>
    <tr>
        <td>Condition Positive (1)</td>
        <td><b>False Negative (FN): 5%</b></td>
        <td><b>True Positive (TP): 45%</b></td>
    </tr>
</table>

- 90% accuracy which is well above the 50% baseline accuracy!
- the precision, recall, and f1 scores are all 0.9.
- it looks great, doesn't it?
- let's rewrite the confusion matrix to reflect rates with respect to the Condition Negative and Condition Positive points!

<table>
    <tr>
        <td colspan="2" rowspan="2"></td>
        <td colspan="2">Predicted class</td>			
    </tr>
    <tr>
        <td>Predicted Negative (0)</td>
        <td>Predicted Positive (1)</td>
    </tr>
    <tr>
        <td rowspan="2">Actual class</td>
        <td>Condition Negative (0) 50% of the points</td>
        <td><b>90% of CNs are correctly classified</b></td>
        <td><b>10% of CNs are incorrectly classified</b></td>
    </tr>
    <tr>
        <td>Condition Positive (1) 50% of the points</td>
        <td><b>10% of CPs are incorrectly classified</b></td>
        <td><b>90% of CPs are correctly classified</b></td>
    </tr>
</table>

## Let's deploy this model

- the incoming data has the same balance as the original dataset (99% to 1%)
- let's assume we have 1e5 new points, 9.9e4 belongs to class 0, 1000 belongs to class 1
- what will be the numbers in the confusion matrix?


<table>
    <tr>
        <td colspan="2" rowspan="2"></td>
        <td colspan="2">Predicted class</td>			
    </tr>
    <tr>
        <td>Predicted Negative (0)</td>
        <td>Predicted Positive (1)</td>
    </tr>
    <tr>
        <td rowspan="2">Actual class</td>
        <td>Condition Negative (0) 99000 points</td>
        <td><b>True Negative (TN): 99000 * 0.9 = 89100 </b></td>
        <td><b>False Positive (FP): 99000 * 0.1 = 9900 </b></td>
    </tr>
    <tr>
        <td>Condition Positive (1) 1000 points</td>
        <td><b>False Negative (FN): 1000 * 0.1 = 100</b></td>
        <td><b>True Positive (TP): 1000 * 0.9 = 900</b></td>
    </tr>
</table>

- the accuracy of this model is still 0.90 but now it is well below the baseline of 0.99!
- recall is good (0.90) but the precision is not great (~0.083)
- the f1 score is ~0.15
- the false positives are overwhelming
- this is why you need to be careful with resampling

# Quiz

# Notes on the midterm presentations

Here are a couple of common issues I've seen:

- **time series**: you have a time series problem *IF* you try to **predict a quantity some dt time in the future**
    - if there is a date time column in your dataset, it does not necessarily mean you have a time series problem!
    - yes, some time-related trends might be present in the dataset but that can be addressed by datetime features and maybe a cyclical encoder
    - Earthquake vs tsunami example
        - If you are trying to predict whether an earthquake is accompanied by a tsunami, that's an IID problem because the earthquakes are independent. It doesn't matter when the last earthquake happened, that will not hae an impact on whether the next earthquake will come with a tsunami or not
        - If you are trying to predict whether an earthquake will happen tomorrow/next week/ next month, that's a time series problem
- **group structure**: group structure needs to be taken into account *IF* there is some sort of **ID column** in your dataset *AND* you are trying to predict a quantity for a **previously unseen ID**
    - if your dataset has a categorical feature, it does not necessarily mean there is group structure in your dataset!
- improve figures!
    - Think about why you show a figure, what is the message you'd like the viewer to take home after looking at the figure
    - Think of ways to bring out that message effectively
    - Use log axis when necessary
    - Do not use the default 10 bins when you create histograms
    - Label the axes and add the units of continuous features
- know your dataset and what you did!
    - You are supposed to be the expert of your data and your project
    - If you can't answer questions, that's a major red flag
    - When that happens during job interviews, you are out
    - think of this from the hiring manager's perspective
    - if you don't know what you did and why, it sends a couple of signals:
        - low involvement - you didn't play a significant role in your previous projects, you don't fully understand the techniques and goals
        - poor communication - maybe you don't know how to explain what you did
        - questionable credibility - does your resume accurately reflect your experiences?
- even if you received max or close to max points, it doesn't necessarily mean that everything you did is correct!
    - If I didn't attend your presentation, come talk to me during my office hours about your project!
    - If you got feedback and comments from me or the TAs, address those for the final presentation and report.
    - The TAs might have missed something either when they graded or when they mentored you.
    - I grade the final reports and if major issues are found in your report, that will have a big impact on your final grade.

# Mudcard